In [3]:
# =========================
# IMPORTS
# =========================
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score

from lightgbm import LGBMClassifier

# =========================
# LOAD DATA
# =========================
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')

X = train.drop(['id', 'Irrigation_Need'], axis=1)
y = train['Irrigation_Need']

X_test = test.drop(['id'], axis=1)

# Encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# =========================
# TARGET ENCODING (SAFE)
# =========================

cat_cols = X.select_dtypes(include=['object']).columns

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for col in cat_cols:
    X[col] = 0
    X_test[col] = 0

    for train_idx, val_idx in skf.split(X, y_encoded):
        X_train_fold = train.iloc[train_idx]
        y_train_fold = y_encoded[train_idx]

        means = X_train_fold.groupby(col)['Irrigation_Need'].apply(
            lambda x: x.map({'Low':0,'Medium':1,'High':2}).mean()
        )

        X.loc[val_idx, col] = train.loc[val_idx, col].map(means)

    # Apply to test
    full_means = train.groupby(col)['Irrigation_Need'].apply(
        lambda x: x.map({'Low':0,'Medium':1,'High':2}).mean()
    )

    X_test[col] = test[col].map(full_means)

# Fill missing
X = X.fillna(0)
X_test = X_test.fillna(0)

# =========================
# MODEL TRAINING
# =========================

test_preds = np.zeros((X_test.shape[0], len(le.classes_)))
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_encoded)):
    print(f"\nFold {fold+1}")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    model = LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    val_preds = model.predict(X_val)
    score = balanced_accuracy_score(y_val, val_preds)
    cv_scores.append(score)
    
    print(f"Balanced Accuracy: {score}")
    
    test_preds += model.predict_proba(X_test) / skf.n_splits

print("\nMean CV Score:", np.mean(cv_scores))

# =========================
# SUBMISSION
# =========================
final_preds = np.argmax(test_preds, axis=1)
final_preds = le.inverse_transform(final_preds)

submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': final_preds
})

submission.to_csv('submission.csv', index=False)

print("Submission created!")

/tmp/ipykernel_55/1484536996.py:48: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.46203107 0.42918808 0.46203107 ... 0.42918808 0.44830224 0.44830224]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X.loc[val_idx, col] = train.loc[val_idx, col].map(means)
/tmp/ipykernel_55/1484536996.py:48: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.43549071 0.45068373 0.42435836 ... 0.43549071 0.43549071 0.42435836]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X.loc[val_idx, col] = train.loc[val_idx, col].map(means)
/tmp/ipykernel_55/1484536996.py:48: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.75758729 0.75758729 0.75758729 ... 0.75758729 0.74066832 0.12960


Fold 1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.057102 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2800
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 19
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
Balanced Accuracy: 0.9678834454240176

Fold 2
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.055992 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2801
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 19
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
Balanced Accuracy: 0.9678808614713527

Fol